# Data Mining Lab Assignment 2 — Book Recommender on a Medallion Data Lake

**Pipeline:** Raw CSVs → 🥉 Bronze (untouched Parquet) → 🥈 Silver (cleaned) → 🥇 Gold (model-ready) → Item-based Collaborative Filtering

**Dataset:** Book-Crossing (Users, Books, Book-Ratings) · **Environment:** Google Colab

| Layer | Folder | Contents |
|---|---|---|
| Bronze | `bronze/` | Raw tables + `ingestion_timestamp`, no cleaning |
| Silver | `silver/` | Deduplicated, validated, typed, filtered tables |
| Gold | `gold/` | Sparse rating matrix, book lookup, user/book aggregates |

## Step 1 — Set up the data lake and build the Bronze layer

In [ ]:
import os, re, io, zipfile, html, string, urllib.request
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse

# Create the three lake layers (plus a landing folder for the raw download)
for folder in ["raw", "bronze", "silver", "gold"]:
    os.makedirs(folder, exist_ok=True)
print("Folders:", [f for f in ["raw", "bronze", "silver", "gold"] if os.path.isdir(f)])

In [ ]:
# Download the Book-Crossing zip and extract the 3 raw CSVs into raw/
URL = "https://cdn.freecodecamp.org/project-data/books/book-crossings.zip"

if not os.path.exists("raw/BX-Users.csv"):
    urllib.request.urlretrieve(URL, "book-crossings.zip")
    with zipfile.ZipFile("book-crossings.zip") as z:
        for name in z.namelist():
            if name.endswith(".csv"):
                # flatten any inner folder so the files land directly in raw/
                with z.open(name) as src, open(os.path.join("raw", os.path.basename(name)), "wb") as dst:
                    dst.write(src.read())

print(sorted(os.listdir("raw")))

In [ ]:
# Load EXACTLY as-is.
#  - sep=";"              : files are semicolon separated
#  - encoding="latin-1"   : files are not UTF-8
#  - dtype=str            : no type coercion; Bronze keeps the raw text (also avoids mixed-type Parquet errors)
#  - on_bad_lines="skip"  : a handful of BX-Books rows are structurally broken (stray quotes/semicolons)
#  - escapechar="\\"      : the files escape quotes with a backslash
READ_KW = dict(sep=";", encoding="latin-1", dtype=str, on_bad_lines="skip", escapechar="\\")

raw_files = {
    "users":   "raw/BX-Users.csv",
    "books":   "raw/BX-Books.csv",
    "ratings": "raw/BX-Book-Ratings.csv",
}

ingestion_ts = pd.Timestamp.now(tz="UTC")     # one timestamp for the whole ingestion run

for name, path in raw_files.items():
    df = pd.read_csv(path, **READ_KW)
    df["ingestion_timestamp"] = ingestion_ts  # the ONLY change allowed in Bronze
    df.to_parquet(f"bronze/{name}.parquet", index=False)
    print(f"bronze/{name}.parquet  -> {df.shape[0]:>9,} rows x {df.shape[1]} cols")

## Step 2 — Explore the Bronze data (EDA)

Each block prints the evidence first, then a one-line markdown note: **what I found → what I'll do in Silver**.
(Numbers quoted in the notes are for the full Book-Crossing dataset; the cell output above each note is the source of truth.)

In [ ]:
users_b   = pd.read_parquet("bronze/users.parquet")
books_b   = pd.read_parquet("bronze/books.parquet")
ratings_b = pd.read_parquet("bronze/ratings.parquet")

for name, df in [("USERS", users_b), ("BOOKS", books_b), ("RATINGS", ratings_b)]:
    print(f"===== {name}  shape={df.shape} =====")
    summary = pd.DataFrame({"dtype": df.dtypes.astype(str),
                            "missing": df.isna().sum(),
                            "missing_%": (df.isna().mean() * 100).round(2)})
    print(summary, "\n")

**2.1 Shape, dtypes, missing values** — Everything is `object` (raw text, by design) and the three tables have ≈ 279k users, ≈ 271k books and ≈ 1.15M ratings; `Age` is missing for ~40% of users and `Book-Author`/`Publisher` have a handful of nulls → **in Silver I will cast to proper numeric types, impute age with the median and drop rows missing key fields (id, title).**

In [ ]:
# Numeric views for EDA only (Bronze on disk stays untouched)
r = ratings_b.copy()
r["rating"] = pd.to_numeric(r["Book-Rating"], errors="coerce")

print("Rating value counts:")
vc = r["rating"].value_counts().sort_index()
print(vc.to_string())
print(f"\nZero ratings: {(r['rating'] == 0).sum():,} of {len(r):,}  ({(r['rating'] == 0).mean():.1%})")

ax = vc.plot(kind="bar", figsize=(7, 3), color="#4C72B0")
ax.set_title("Rating distribution (Bronze)"); ax.set_xlabel("rating"); ax.set_ylabel("count")
plt.tight_layout(); plt.show()

**2.2 Ratings distribution** — About 60%+ of all rows are `0`, which means "implicit / no explicit rating", not a score of zero, and the explicit ratings are skewed towards 7–10 → **in Silver I will drop all 0 ratings and keep only 1–10.**

In [ ]:
u = users_b.copy()
u["age_num"] = pd.to_numeric(u["Age"], errors="coerce")
print(u["age_num"].describe().round(1).to_string())
print(f"\nAge < 5   : {(u['age_num'] < 5).sum():,}")
print(f"Age > 100 : {(u['age_num'] > 100).sum():,}")
print(f"Age null  : {u['age_num'].isna().sum():,}")

b = books_b.copy()
b["year_num"] = pd.to_numeric(b["Year-Of-Publication"], errors="coerce")
cur_year = datetime.now().year
print(f"\nPublication year: non-numeric/null = {b['year_num'].isna().sum():,}")
print(f"Year == 0        : {(b['year_num'] == 0).sum():,}")
print(f"Year > {cur_year}     : {(b['year_num'] > cur_year).sum():,}")
print(f"Year < 1400 (>0) : {((b['year_num'] > 0) & (b['year_num'] < 1400)).sum():,}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
u.loc[u["age_num"].between(0, 120), "age_num"].plot(kind="hist", bins=60, ax=ax[0], color="#55A868")
ax[0].set_title("Age (0-120 shown)")
b.loc[b["year_num"].between(1800, 2100), "year_num"].plot(kind="hist", bins=60, ax=ax[1], color="#C44E52")
ax[1].set_title("Publication year (1800-2100 shown)")
plt.tight_layout(); plt.show()

**2.3 User ages** — Ages contain impossible values (0–4 and up to 200+) plus ~40% nulls → **in Silver: set age < 5 or > 100 to null, then fill nulls with the median age.**

**2.4 Publication years** — Thousands of books have year `0` and a few have future years (e.g. 2030–2050); some rows are non-numeric because of shifted columns → **in Silver: coerce to numeric and set 0 / future / non-numeric years to null (not imputed, year isn't used by the model).**

In [ ]:
per_user = r.groupby("User-ID").size()
per_book = r.groupby("ISBN").size()

for label, s in [("ratings per user", per_user), ("ratings per book", per_book)]:
    print(f"--- {label} ---")
    print(s.describe(percentiles=[.5, .75, .9, .99]).round(1).to_string())
    print(f"share with < 10 ratings: {(s < 10).mean():.1%}\n")

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
for a, s, t in [(ax[0], per_user, "Ratings per user"), (ax[1], per_book, "Ratings per book")]:
    a.hist(np.log10(s), bins=40, color="#8172B2"); a.set_title(t + " (log10 x-axis)"); a.set_xlabel("log10(#ratings)")
plt.tight_layout(); plt.show()

**2.5 Ratings per user / per book** — Both are extremely long-tailed: the median user and the median book have only 1–3 ratings, and ~90% of users/books have fewer than 10 → **in Silver I will apply a minimum-ratings threshold (chosen with the table in Step 3) so similarities rest on enough evidence.**

In [ ]:
def sparsity(df_ratings, ucol, icol):
    n_u, n_i, n_r = df_ratings[ucol].nunique(), df_ratings[icol].nunique(), len(df_ratings)
    return n_u, n_i, n_r, 100 * (1 - n_r / (n_u * n_i))

for label, d in [("All ratings (incl. 0)", r), ("Explicit only (1-10)", r[r["rating"] > 0])]:
    n_u, n_i, n_r, sp = sparsity(d, "User-ID", "ISBN")
    print(f"{label:24s}: {n_u:>7,} users x {n_i:>7,} books, {n_r:>9,} ratings -> density {100-sp:.5f}%  (sparsity {sp:.4f}%)")

**2.6 Matrix sparsity** — Only ~0.001–0.003% of the possible user–book pairs have a rating (≥ 99.99% empty) → **in Gold I will store the matrix as a SciPy sparse CSR matrix (a dense one would need hundreds of GB) and use a sparse-friendly cosine kNN in Step 5.**

## Step 3 — Clean the data and build the Silver layer

In [ ]:
users   = pd.read_parquet("bronze/users.parquet").drop(columns="ingestion_timestamp")
books   = pd.read_parquet("bronze/books.parquet").drop(columns="ingestion_timestamp")
ratings = pd.read_parquet("bronze/ratings.parquet").drop(columns="ingestion_timestamp")

# consistent snake_case column names
users   = users.rename(columns={"User-ID": "user_id", "Location": "location", "Age": "age"})
books   = books.rename(columns={"ISBN": "isbn", "Book-Title": "title", "Book-Author": "author",
                                "Year-Of-Publication": "year", "Publisher": "publisher"})
ratings = ratings.rename(columns={"User-ID": "user_id", "ISBN": "isbn", "Book-Rating": "rating"})

log = {}   # row-count audit trail
def track(step, **tables):
    log[step] = {k: len(v) for k, v in tables.items()}
track("0 bronze", users=users, books=books, ratings=ratings)

**3.1 Remove duplicates** (exact duplicate rows, then duplicate primary keys)

In [ ]:
users   = users.drop_duplicates().drop_duplicates(subset="user_id", keep="first")
books   = books.drop_duplicates().drop_duplicates(subset="isbn", keep="first")
ratings = ratings.drop_duplicates().drop_duplicates(subset=["user_id", "isbn"], keep="last")  # one rating per (user, book)
track("1 dedupe", users=users, books=books, ratings=ratings)

**3.2 Ratings: cast to numbers, drop 0 ("no explicit rating"), keep 1–10**

In [ ]:
ratings["user_id"] = pd.to_numeric(ratings["user_id"], errors="coerce")
ratings["rating"]  = pd.to_numeric(ratings["rating"],  errors="coerce")
ratings = ratings.dropna(subset=["user_id", "isbn", "rating"])
ratings = ratings[ratings["rating"].between(1, 10)].copy()
ratings["user_id"] = ratings["user_id"].astype("int64")
ratings["rating"]  = ratings["rating"].astype("int8")
track("2 ratings 1-10", users=users, books=books, ratings=ratings)

**3.3 User ages:** `<5` or `>100` → null → fill with the median. **3.4 Location → city / state / country**

In [ ]:
users["user_id"] = pd.to_numeric(users["user_id"], errors="coerce")
users = users.dropna(subset=["user_id"]).copy()
users["user_id"] = users["user_id"].astype("int64")

# ---- age
users["age"] = pd.to_numeric(users["age"], errors="coerce")
users.loc[(users["age"] < 5) | (users["age"] > 100), "age"] = np.nan
median_age = users["age"].median()
users["age"] = users["age"].fillna(median_age).astype("int16")
print("median age used for imputation:", median_age)

# ---- location: "city, state, country"
loc = users["location"].fillna("").str.split(",", n=2, expand=True)
for c in range(3):
    if c not in loc.columns: loc[c] = None
users["city"], users["state"], users["country"] = loc[0], loc[1], loc[2]
users = users.drop(columns="location")
users[["city", "state", "country"]].head()

**3.5 Publication year:** `0` or future → null. **3.6 Text cleaning** (strip, collapse spaces, decode HTML entities such as `&amp;`, standardise case) and drop the image-URL columns

In [ ]:
# ---- publication year
books["year"] = pd.to_numeric(books["year"], errors="coerce")
books.loc[(books["year"] == 0) | (books["year"] > datetime.now().year), "year"] = np.nan
books["year"] = books["year"].astype("Int16")

# ---- text columns
def clean_text(s: pd.Series, case: str = "keep") -> pd.Series:
    s = (s.astype("string")
           .map(lambda x: html.unescape(x) if isinstance(x, str) else x)   # &amp; -> &
           .str.replace(r"\s+", " ", regex=True)                           # collapse runs of whitespace
           .str.strip())
    s = s.replace({"": pd.NA})
    if case == "lower":
        s = s.str.lower()
    elif case == "title":
        s = s.str.lower().map(lambda x: string.capwords(x) if isinstance(x, str) else x)
    return s

books["isbn"]      = clean_text(books["isbn"]).str.upper()
ratings["isbn"]    = clean_text(ratings["isbn"]).str.upper()
books["title"]     = clean_text(books["title"],     "title")
books["author"]    = clean_text(books["author"],    "title")
books["publisher"] = clean_text(books["publisher"], "title")
for c in ["city", "state", "country"]:
    users[c] = clean_text(users[c], "lower")        # lower-case: "USA" == "usa"
    users[c] = users[c].replace({"n/a": pd.NA})

# ---- drop image-URL columns (not needed for the model)
books = books.drop(columns=[c for c in books.columns if c.lower().startswith("image")])
books = books.dropna(subset=["isbn", "title"])
print(books.columns.tolist())
track("3 clean cols", users=users, books=books, ratings=ratings)

**3.7 Referential integrity** — keep only ratings whose user *and* book exist in the users / books tables

In [ ]:
ratings = ratings[ratings["user_id"].isin(users["user_id"]) & ratings["isbn"].isin(books["isbn"])].copy()
track("4 referential integrity", users=users, books=books, ratings=ratings)
print(f"{len(ratings):,} ratings left, {ratings.user_id.nunique():,} users, {ratings.isbn.nunique():,} books")

**3.8 Minimum-ratings threshold** — chosen from the EDA, not guessed.

Because the filter on users changes book counts (and vice-versa), I apply it repeatedly until nothing changes (a "k-core"). The table below shows what each candidate threshold would leave.

In [ ]:
def k_core(r, min_user, min_book):
    '''Iteratively drop users/books below the thresholds until the set is stable.'''
    while True:
        n0 = len(r)
        r = r[r.groupby("user_id")["isbn"].transform("size") >= min_user]
        r = r[r.groupby("isbn")["user_id"].transform("size") >= min_book]
        if len(r) == n0:
            return r

rows = []
for k in [1, 3, 5, 10, 20, 50]:
    kc = k_core(ratings, k, k)
    n_u, n_b = kc.user_id.nunique(), kc.isbn.nunique()
    rows.append({"min ratings (user & book)": k, "users": n_u, "books": n_b, "ratings": len(kc),
                 "% of ratings kept": round(100 * len(kc) / len(ratings), 1),
                 "avg ratings / book": round(len(kc) / max(n_b, 1), 1),
                 "density %": round(100 * len(kc) / max(n_u * n_b, 1), 3)})
pd.DataFrame(rows).set_index("min ratings (user & book)")

In [ ]:
MIN_USER_RATINGS = 10
MIN_BOOK_RATINGS = 10

ratings = k_core(ratings, MIN_USER_RATINGS, MIN_BOOK_RATINGS).reset_index(drop=True)
users = users[users["user_id"].isin(ratings["user_id"])].reset_index(drop=True)
books = books[books["isbn"].isin(ratings["isbn"])].reset_index(drop=True)
track("5 min-ratings filter", users=users, books=books, ratings=ratings)

print(pd.DataFrame(log).T.to_string())

**Threshold decision: users ≥ 10 ratings and books ≥ 10 ratings.**
*Justification from the EDA:* ~90% of users and books have fewer than 10 ratings, so most of the matrix is noise (a book with 1–2 raters gives a cosine similarity of 0 or 1 by chance). A threshold of 10 removes that tail while leaving a dense core with ≥ 10 co-raters per book on average; the table shows that going to 20 or 50 throws away most of the books (hurting catalogue coverage) for little density gain, while 3–5 keeps too many one-off items.

In [ ]:
users.to_parquet("silver/users.parquet", index=False)
books.to_parquet("silver/books.parquet", index=False)
ratings.to_parquet("silver/ratings.parquet", index=False)

print(os.listdir("silver"))
display(users.head(3)); display(books.head(3)); display(ratings.head(3))

## Step 4 — Build the Gold layer

In [ ]:
users   = pd.read_parquet("silver/users.parquet")
books   = pd.read_parquet("silver/books.parquet")
ratings = pd.read_parquet("silver/ratings.parquet")

# ---- stable integer indices for matrix rows (users) and columns (books)
user_index = pd.DataFrame({"user_id": np.sort(ratings["user_id"].unique())})
user_index["user_idx"] = np.arange(len(user_index))
book_index = pd.DataFrame({"isbn": np.sort(ratings["isbn"].unique())})
book_index["book_idx"] = np.arange(len(book_index))

r = (ratings.merge(user_index, on="user_id").merge(book_index, on="isbn"))

# ---- 1) sparse user x book rating matrix (CSR)
rating_matrix = sparse.csr_matrix(
    (r["rating"].astype("float32"), (r["user_idx"], r["book_idx"])),
    shape=(len(user_index), len(book_index)))
sparse.save_npz("gold/rating_matrix.npz", rating_matrix)
user_index.to_parquet("gold/user_index.parquet", index=False)
book_index.to_parquet("gold/book_index.parquet", index=False)

density = rating_matrix.nnz / (rating_matrix.shape[0] * rating_matrix.shape[1])
print(f"matrix {rating_matrix.shape}, nnz={rating_matrix.nnz:,}, density={density:.4%}")
print(f"sparse size ~{(rating_matrix.data.nbytes + rating_matrix.indices.nbytes + rating_matrix.indptr.nbytes)/1e6:.1f} MB "
      f"vs dense ~{rating_matrix.shape[0]*rating_matrix.shape[1]*4/1e9:.2f} GB")

In [ ]:
# ---- 2) book lookup table (id, title, author)
book_lookup = (book_index.merge(books[["isbn", "title", "author", "year", "publisher"]], on="isbn", how="left")
                         [["book_idx", "isbn", "title", "author", "year", "publisher"]])
book_lookup.to_parquet("gold/book_lookup.parquet", index=False)

# ---- 3) user-level and book-level aggregates
user_stats = (r.groupby("user_id")["rating"].agg(avg_rating="mean", n_ratings="size").round(3).reset_index())
book_stats = (r.groupby("isbn")["rating"].agg(avg_rating="mean", n_ratings="size").round(3).reset_index()
                .merge(book_lookup[["isbn", "title", "author"]], on="isbn"))
user_stats.to_parquet("gold/user_stats.parquet", index=False)
book_stats.to_parquet("gold/book_stats.parquet", index=False)

print(sorted(os.listdir("gold")))
display(book_lookup.head()); display(user_stats.head()); display(book_stats.sort_values("n_ratings", ascending=False).head())

## Step 5 — Item-based collaborative filtering

Idea: two books are similar if the *same users* rated them in a similar way. I represent each book as its column of user ratings (sparse), compute **cosine distance** with `NearestNeighbors`, and recommend books that are close to the ones a user already liked.

**5.1 Load the Gold matrix and make rows = books, columns = users**

In [ ]:
from sklearn.neighbors import NearestNeighbors

rating_matrix = sparse.load_npz("gold/rating_matrix.npz").tocsr()     # users x books
user_index    = pd.read_parquet("gold/user_index.parquet")
book_lookup   = pd.read_parquet("gold/book_lookup.parquet").sort_values("book_idx").reset_index(drop=True)
book_stats    = pd.read_parquet("gold/book_stats.parquet")

book_user = rating_matrix.T.tocsr()          # books x users  (each row = one book)
print("book x user matrix:", book_user.shape)

lookup = book_lookup.merge(book_stats[["isbn", "n_ratings", "avg_rating"]], on="isbn")
lookup = lookup.sort_values("book_idx").reset_index(drop=True)   # position == book_idx
lookup["title_key"] = lookup["title"].str.lower()
user_pos = pd.Series(user_index["user_idx"].values, index=user_index["user_id"].values)

**5.2 Cosine similarity between books** (brute-force cosine kNN works directly on the sparse matrix)

In [ ]:
knn = NearestNeighbors(metric="cosine", algorithm="brute", n_jobs=-1)
knn.fit(book_user)
print("fitted on", book_user.shape[0], "books")

**5.3 `similar_books(title)` — the 10 most similar books**

In [ ]:
def _find_book(title: str):
    '''Case-insensitive exact match, falling back to substring match. If several editions
    match, use the one with the most ratings.'''
    t = title.strip().lower()
    hits = lookup[lookup["title_key"] == t]
    if hits.empty:
        hits = lookup[lookup["title_key"].str.contains(re.escape(t), na=False)]
    if hits.empty:
        return None
    return hits.sort_values("n_ratings", ascending=False).iloc[0]

def similar_books(title: str, n: int = 10) -> pd.DataFrame:
    book = _find_book(title)
    if book is None:
        raise ValueError(f"No book matching '{title}' in the catalogue (only books with >= {MIN_BOOK_RATINGS} ratings are indexed).")
    k = min(n * 5 + 1, book_user.shape[0])               # over-fetch: other editions get filtered out
    dist, idx = knn.kneighbors(book_user[book["book_idx"]], n_neighbors=k)
    out = lookup.iloc[idx[0]][["book_idx", "title", "author", "n_ratings"]].copy()
    out["similarity"] = (1 - dist[0]).round(4)
    out = out[(out["book_idx"] != book["book_idx"]) & (out["title"] != book["title"])]
    out = out.drop_duplicates("title").head(n)
    print(f"Books similar to: {book['title']}  —  {book['author']}")
    return out[["title", "author", "similarity", "n_ratings"]].reset_index(drop=True)

popular_title = lookup.sort_values("n_ratings", ascending=False).iloc[0]["title"]
similar_books(popular_title)

**5.4 `recommend(user_id, n=10)`**

1. Take the books the user rated highly (rating ≥ 7; if there are none, fall back to their 5 best-rated books).
2. For each liked book, fetch its nearest neighbour books.
3. Score each candidate: **Σ similarity × the user's rating of the liked book** (a candidate close to several highly-rated books accumulates a bigger score).
4. Remove books the user has already rated (also other editions of the same title), rank, return the top *n*.

In [ ]:
def liked_books(user_id: int, min_rating: int = 7, max_books: int = 20) -> pd.DataFrame:
    '''Books the user rated highly (highest first).'''
    row = rating_matrix[user_pos[user_id]]
    d = pd.DataFrame({"book_idx": row.indices, "rating": row.data.astype(int)})
    liked = d[d["rating"] >= min_rating]
    if liked.empty:
        liked = d.sort_values("rating", ascending=False).head(5)
    liked = liked.sort_values("rating", ascending=False).head(max_books)
    return liked.merge(lookup[["book_idx", "title", "author"]], on="book_idx").reset_index(drop=True)

def recommend(user_id: int, n: int = 10, min_rating: int = 7, k_neighbors: int = 30) -> pd.DataFrame:
    if user_id not in user_pos.index:
        raise ValueError(f"user {user_id} not in the model (users need >= {MIN_USER_RATINGS} explicit ratings).")

    liked = liked_books(user_id, min_rating)
    seen_idx = set(rating_matrix[user_pos[user_id]].indices)               # everything the user already rated
    seen_titles = set(lookup.loc[list(seen_idx), "title_key"])            # ...including other editions

    dist, idx = knn.kneighbors(book_user[liked["book_idx"].values], n_neighbors=k_neighbors + 1)
    scores = {}
    for row_i, user_rating in enumerate(liked["rating"].values):
        for d, j in zip(dist[row_i], idx[row_i]):
            if j in seen_idx:
                continue
            scores[j] = scores.get(j, 0.0) + (1.0 - d) * user_rating      # similarity weighted by the user's rating

    rec = pd.DataFrame({"book_idx": list(scores.keys()), "score": list(scores.values())})
    rec = rec.merge(lookup[["book_idx", "title", "author", "title_key", "n_ratings", "avg_rating"]], on="book_idx")
    rec = rec[~rec["title_key"].isin(seen_titles)]
    rec = rec.sort_values("score", ascending=False).drop_duplicates("title").head(n)
    return rec[["title", "author", "score", "avg_rating", "n_ratings"]].round(3).reset_index(drop=True)

## Step 6 — Demo and sanity check

Three different users (random, fixed seed; each has ≥ 20 ratings and at least 3 highly-rated books). For each we print what they liked next to what we recommend.

In [ ]:
user_stats = pd.read_parquet("gold/user_stats.parquet")
candidates = user_stats[user_stats["n_ratings"] >= 20]["user_id"]
candidates = [u for u in candidates if (liked_books(u, 7)["rating"] >= 7).sum() >= 3]
demo_users = pd.Series(candidates).sample(3, random_state=42).tolist()

for uid in demo_users:
    st = user_stats[user_stats["user_id"] == uid].iloc[0]
    print("=" * 100)
    print(f"USER {uid}  |  {int(st['n_ratings'])} ratings, mean rating {st['avg_rating']:.2f}")
    print("-" * 100)
    print("Books this user rated highly:")
    display(liked_books(uid, 7).head(8)[["title", "author", "rating"]])
    print("Top-10 recommendations:")
    display(recommend(uid, n=10)[["title", "author", "score"]])

### Sanity check — do the recommendations make sense?

For each demo user, compare the two tables: recommended books should share **author, series or genre** with the liked books (e.g. readers of one thriller author get similar thrillers), none of the recommendations should already be in the user's history, and the scores should decrease down the list. Item-based CF only uses co-rating behaviour (no content features), so occasional best-sellers appear for everyone — a known popularity bias.

In [ ]:
# Automatic checks
for uid in demo_users:
    rec = recommend(uid, 10)
    seen_titles = set(lookup.loc[list(rating_matrix[user_pos[uid]].indices), "title"])
    assert len(rec) == 10,                                  f"user {uid}: expected 10 recs, got {len(rec)}"
    assert not (set(rec["title"]) & seen_titles),           f"user {uid}: recommended an already-read book"
    assert rec["score"].is_monotonic_decreasing,            f"user {uid}: scores not sorted"
    assert rec["title"].is_unique,                          f"user {uid}: duplicate titles"
print("All sanity checks passed ✅")

## Summary & limitations

* **Pipeline:** raw CSV → Bronze (raw + `ingestion_timestamp`) → Silver (dedupe, 1–10 ratings, age/year fixes, location split, text clean, referential integrity, ≥ 10 ratings) → Gold (sparse CSR matrix, book lookup, user/book aggregates).
* **Model:** item-based CF, cosine similarity on the sparse book × user matrix, `NearestNeighbors(metric="cosine")`; user recommendations = Σ(similarity × user rating) over the user's liked books, excluding already-read books.
* **Limitations:** cold-start (users/books below the threshold are not in the model), popularity bias, and different editions of the same title have different ISBNs (handled only by title de-duplication at output time).